## Instalação de pacotes que serão utilizados pelo Python e não estão nativamente no ambiente da Databricks

In [0]:
%pip install polars==1.0.0 --quiet
%pip install duckdb==1.2.1 --quiet
%pip install odfpy --quiet
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Configurações iniciais de módulos, diretórios e URLs com os dados que deverão ser coletados para posterior rotina de dados.

Os dados serão utilizados de repositórios públicos da **Anatel** e **IBGE**

In [0]:
import polars as pl
import pandas as pd
import duckdb as db
from duckdb.typing import *
import os, shutil, sys
from pathlib import Path 
import zipfile
import tempfile
import subprocess as sub
import numpy as np
from datetime import *
from dateutil.relativedelta import relativedelta
import unicodedata
import re

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

PROJ_PATH = '/Workspace/Users/vini.leodido@gmail.com'
STG_PATH = os.path.join(PROJ_PATH, '_temp')
DATABASE = os.path.join(STG_PATH, 'database')
TMP_FILES = os.path.join(STG_PATH, 'tmp_files')

if not os.path.exists(STG_PATH):
    os.mkdir(STG_PATH)

URL_RQUAL_IND = 'https://www.anatel.gov.br/dadosabertos/paineis_de_dados/qualidade/indicadores_rqual.zip'
URL_ESTACOES_SMP = 'https://www.anatel.gov.br/dadosabertos/paineis_de_dados/outorga_e_licenciamento/estacoes_smp.zip'
URL_IBGE_MUNICIPIOS = 'https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/divisao_territorial/2023/DTB_2023.zip'
URL_ANATEL_AREAS_LOCAIS = 'https://www.anatel.gov.br/dadosabertos/paineis_de_dados/areastarifarias/areaslocais.zip'

## Funções auxiliares que serão utilizadas para ajudar na normalização e transformação dos dados.

In [0]:
#Leitura do valor em Bytes para "HumanReadable"
def sizeof_fmt(num, suffix="B"):
    for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Yi{suffix}"

#Coleta Dados URL
def get_file_url(TMP_FILES, URL_NAME, proxy=False):
    name_file = os.path.basename(URL_NAME)
    file_path = os.path.join(TMP_FILES,name_file)
    r = requests.get(URL_NAME, stream=True, verify=False, proxies=(proxy_dict if proxy else {}))
    if r.ok:
        print("saving to", os.path.abspath(file_path))
        with open(file_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 8):
                if chunk:
                    f.write(chunk)
                    f.flush()
                    os.fsync(f.fileno())
    else:  # HTTP status code 4XX/5XX
        print("Download failed: status code {}\n{}".format(r.status_code, r.text))


# Funções auxiliares para limpar nomes de colunas
def remove_accents(text):
    return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')

def only_alphanum(text):
    return re.sub(r'[^a-zA-Z0-9 ]', ' ', text)

def non_alphanum_underscore(text):
    return re.sub(r'\s+', '_', text.strip())

def dashes2underscore(text):
    return re.sub(r'[-]', '_', text.strip())


for v_paths in [DATABASE, TMP_FILES]:
    if os.path.exists(v_paths):
        shutil.rmtree(v_paths)
        os.mkdir(v_paths)
    else:
        os.mkdir(v_paths)

#definição do apontamento e diretório onde serão armazenados o metadados do duckdb
Path(DATABASE).mkdir(parents=True, exist_ok=True)
db_file_path = os.path.join(DATABASE, "db_portalanatel_ind_rqual.duckdb")

## Obtendo os dados pelas suas respectivas URLs de acesso

In [0]:
#get_file_url(TMP_FILES, URL_RQUAL_IND)
#get_file_url(TMP_FILES, URL_ESTACOES_SMP)
#get_file_url(TMP_FILES, URL_IBGE_MUNICIPIOS)
#get_file_url(TMP_FILES, URL_ANATEL_AREAS_LOCAIS)

## Transformação e pré-processamento dos dados de múnicipios brasileiros obtidos pelo **IBGE**

In [0]:
#coleta arquivo
get_file_url(TMP_FILES, URL_IBGE_MUNICIPIOS)

#Arquivos Base Municípios IBGE
filename_ibge = os.path.join(TMP_FILES,os.path.basename(URL_IBGE_MUNICIPIOS))
with zipfile.ZipFile(filename_ibge, 'r') as zip_ref:
    source = zip_ref.open('DTB_2023/RELATORIO_DTB_BRASIL_MUNICIPIO.ods')
    target = open(os.path.join(TMP_FILES, 'RELATORIO_DTB_BRASIL_MUNICIPIO.ods'), "wb")
    with source, target:
            shutil.copyfileobj(source, target)
df_dtb_municipio = pd.read_excel(os.path.join(TMP_FILES,"RELATORIO_DTB_BRASIL_MUNICIPIO.ods"), skiprows=6)
df_dtb_municipio.head()

#Normaliza nome das Colunas encontradas
tmp_cols = []
for col in df_dtb_municipio.columns.to_list():
    tmp_cols.append(non_alphanum_underscore(only_alphanum(remove_accents(col.upper()))))
df_dtb_municipio.columns = tmp_cols; del tmp_cols

conn = db.connect(db_file_path)
conn.sql("DROP TABLE IF EXISTS DTB_IBGE_MUNICIPIOS")
conn.sql("CREATE TABLE DTB_IBGE_MUNICIPIOS AS SELECT * FROM df_dtb_municipio")
conn.close()

os.remove(os.path.join(TMP_FILES,os.path.basename(URL_IBGE_MUNICIPIOS)))

saving to /Workspace/Users/vini.leodido@gmail.com/_temp/tmp_files/DTB_2023.zip


### Transformação e pré-processamento dos dados disponibilizados pela **Anatel**

In [0]:
#coleta arquivo
get_file_url(TMP_FILES, URL_ANATEL_AREAS_LOCAIS)

#Arquivos Anatel - Códigos de Areas Locais
filename_cn = os.path.join(TMP_FILES,os.path.basename(URL_ANATEL_AREAS_LOCAIS))

with zipfile.ZipFile(filename_cn, 'r') as zip_ref:
    source = zip_ref.open('CODIGOS_NACIONAIS_PGCN.csv')
    target = open(os.path.join(TMP_FILES, 'CODIGOS_NACIONAIS_PGCN.csv'), "wb")
    with source, target:
            shutil.copyfileobj(source, target)
df_arealocal = pd.read_csv(os.path.join(TMP_FILES,"CODIGOS_NACIONAIS_PGCN.csv"), sep=';')
df_arealocal.head()

#Normaliza nome das Colunas encontradas
tmp_cols = []
for col in df_arealocal.columns.to_list():
    tmp_cols.append(dashes2underscore(non_alphanum_underscore(only_alphanum(remove_accents(col.upper())))))
df_arealocal.columns = tmp_cols; del tmp_cols

conn = db.connect(db_file_path)
conn.sql("DROP TABLE IF EXISTS AREALOCAL")
conn.sql("CREATE TABLE AREALOCAL AS SELECT * FROM df_arealocal")
conn.close()

os.remove(os.path.join(TMP_FILES,os.path.basename(URL_ANATEL_AREAS_LOCAIS)))

saving to /Workspace/Users/vini.leodido@gmail.com/_temp/tmp_files/areaslocais.zip


In [0]:
# coleta arquivo
get_file_url(TMP_FILES, URL_RQUAL_IND)

# Arquivos Base Anatel - Indicadores
filename_indicadores = os.path.join(TMP_FILES, os.path.basename(URL_RQUAL_IND))

# Lê e processa diretamente do ZIP
with zipfile.ZipFile(filename_indicadores, 'r') as zip_ref:
    csv_filename = zip_ref.namelist()[0]
    
    # Cria um arquivo temporário em memória
    with zip_ref.open(csv_filename) as csv_file:
        # Cria arquivo temporário para o Polars processar
        with tempfile.NamedTemporaryFile(mode='wb', suffix='.csv', delete=False) as temp_file:
            temp_file.write(csv_file.read())
            temp_csv_path = temp_file.name
        
        try:
            parquet_file = dashes2underscore(os.path.normpath(
                os.path.join(TMP_FILES, csv_filename.replace('.csv', '.parquet'))
            ))
            
            print(f'Tratando o arquivo {csv_filename} diretamente do ZIP')
            
            # Schema inference
            df_schema = pl.scan_csv(
                temp_csv_path,
                separator=';',
                encoding='utf8',
                has_header=True,
                infer_schema_length=10000,
                null_values=['NÃO IDENTIFICADA'],
                n_rows=100000
            ).collect_schema()
            
            # Normaliza colunas
            tmp_dict = {}
            for k, v in df_schema.items():
                tmp_dict[non_alphanum_underscore(only_alphanum(remove_accents(k))).upper()] = df_schema[k]
            df_schema = tmp_dict.copy()
            del tmp_dict
            
            # Conversão LazyFrame para Parquet
            pl.scan_csv(
                temp_csv_path,
                separator=';',
                encoding='utf8',
                has_header=True,
                infer_schema_length=10000,
                null_values=['NÃO IDENTIFICADA'],
                schema=df_schema
            ).sink_parquet(
                parquet_file,
                compression="zstd",
                row_group_size=100_000
            )
            
            print(f'Arquivo parquet gerado {os.path.basename(parquet_file)} | tamanho do arquivo {sizeof_fmt(os.path.getsize(parquet_file))}')
            
        finally:
            # Limpa o arquivo temporário
            os.unlink(temp_csv_path)

# Resto do código permanece igual...
conn = db.connect(db_file_path)
tbl_name = non_alphanum_underscore(os.path.splitext(os.path.basename(parquet_file))[0].upper())
conn.execute(f"DROP TABLE IF EXISTS {tbl_name}")
conn.execute(f"CREATE TABLE {tbl_name} AS SELECT * FROM parquet_scan('{parquet_file}')")
conn.close()

os.remove(filename_indicadores)

saving to /Workspace/Users/vini.leodido@gmail.com/_temp/tmp_files/indicadores_rqual.zip
Tratando o arquivo Tabela_CSV_Indicadores_RQUAL.csv diretamente do ZIP
Arquivo parquet gerado Tabela_CSV_Indicadores_RQUAL.parquet | tamanho do arquivo 180.4MiB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [0]:
# coleta arquivo
get_file_url(TMP_FILES, URL_ESTACOES_SMP)

# Arquivos Base Anatel - Estações Serviço Móvel - ERBs
filename_base_sites = os.path.join(TMP_FILES, os.path.basename(URL_ESTACOES_SMP))

# Lê e processa diretamente do ZIP
with zipfile.ZipFile(filename_base_sites, 'r') as zip_ref:
    csv_filename = zip_ref.namelist()[0]
    
    # Cria um arquivo temporário para o Polars processar
    with zip_ref.open(csv_filename) as csv_file:
        with tempfile.NamedTemporaryFile(mode='wb', suffix='.csv', delete=False) as temp_file:
            temp_file.write(csv_file.read())
            temp_csv_path = temp_file.name
        
        try:
            parquet_file = dashes2underscore(os.path.normpath(
                os.path.join(TMP_FILES, csv_filename.replace('.csv', '.parquet'))
            ))
            
            print(f'Tratando o arquivo {csv_filename} diretamente do ZIP')
            
            # Schema inference com parâmetros específicos para estações
            df_schema = pl.scan_csv(
                temp_csv_path,
                separator=';',
                encoding='utf8',
                has_header=True,
                infer_schema_length=400000,  # Valor maior conforme original
                null_values=['NÃO IDENTIFICADA', '#N/A', ','],
                n_rows=400000
            ).collect_schema()
            
            # Normaliza nome das Colunas encontradas
            tmp_dict = {}
            for k, v in df_schema.items():
                tmp_dict[non_alphanum_underscore(only_alphanum(remove_accents(k))).upper()] = df_schema[k]
            df_schema = tmp_dict.copy()
            del tmp_dict
            
            # Conversão LazyFrame para Parquet
            pl.scan_csv(
                temp_csv_path,
                separator=';',
                encoding='utf8',
                has_header=True,
                infer_schema_length=10000,  # Valor menor para processamento final
                null_values=['NÃO IDENTIFICADA', '#N/A', ','],
                schema=df_schema
            ).sink_parquet(
                parquet_file,
                compression="zstd",
                row_group_size=100_000
            )
            
            print(f'Arquivo parquet gerado {os.path.basename(parquet_file)} | tamanho do arquivo {sizeof_fmt(os.path.getsize(parquet_file))}')
            
        finally:
            # Limpa o arquivo temporário
            os.unlink(temp_csv_path)

# Carregando o dataset para Transformações no DuckDB
conn = db.connect(db_file_path)
tbl_name = non_alphanum_underscore(os.path.splitext(os.path.basename(parquet_file))[0].upper())
conn.execute(f"DROP TABLE IF EXISTS {tbl_name}")
conn.execute(f"CREATE TABLE {tbl_name} AS SELECT * FROM parquet_scan('{parquet_file}')")
conn.close()

# Remove apenas o arquivo ZIP original
os.remove(filename_base_sites)

saving to /Workspace/Users/vini.leodido@gmail.com/_temp/tmp_files/estacoes_smp.zip
Tratando o arquivo Estacoes_SMP.csv diretamente do ZIP
Arquivo parquet gerado Estacoes_SMP.parquet | tamanho do arquivo 29.1MiB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [0]:
#Remove diretorio temporario aliviando espaço em disco do servidor
shutil.rmtree(TMP_FILES)

### Processamento via **duckdb** para ajustes pontuais em arquivos.

Optei por utilizar o duckdb como uma camada _Bronze_ , tendo os dados parquet transformado anteriormente em sua camada analítica SQL.

In [0]:
conn = db.connect(db_file_path)

v_sql_smp = '''
DROP VIEW IF EXISTS V_RQUAL_ANATEL_SMP;

CREATE OR REPLACE VIEW V_RQUAL_ANATEL_SMP as
SELECT
 try_cast(MESDATPER as INTEGER) MESDATPER, ANO, MES, REPLACE(SERVICO, 'Telefonia Móvel', 'SMP') SERVICO, PRESTADORA, CODIGO_IBGE, UF, NOME_DO_MUNICIPIO,
 max("IND1_max(RESULTADO)") IND1,
 max("IND2_max(RESULTADO)") IND2,
 max("IND3_max(RESULTADO)") IND3,
 max("IND4_max(RESULTADO)") IND4,
 max("IND5_max(RESULTADO)") IND5,
 max("IND6_max(RESULTADO)") IND6,
 max("IND7_max(RESULTADO)") IND7,
 max("IND8_max(RESULTADO)") IND8,
 least(max("IND4_MAX(NUMERO_MEDIDAS)"), max("IND5_MAX(NUMERO_MEDIDAS)"), max("IND6_MAX(NUMERO_MEDIDAS)"), max("IND7_MAX(NUMERO_MEDIDAS)")) NUMERO_MEDIDAS,
 least(max("IND4_MAX(NUMERO_COLETORES)"), max("IND5_MAX(NUMERO_COLETORES)"), max("IND6_MAX(NUMERO_COLETORES)"), max("IND7_MAX(NUMERO_COLETORES)")) NUMERO_COLETORES,
 case when least(max("IND4_MAX(NUMERO_MEDIDAS)"), max("IND5_MAX(NUMERO_MEDIDAS)"), max("IND6_MAX(NUMERO_MEDIDAS)"), max("IND7_MAX(NUMERO_MEDIDAS)")) >= 109 then 1 else 0 end VALIDADE_ESTATISTICA
from
(
	select
	try_cast(i.ANO as varchar)||case when length(try_cast(i.MES as varchar))=1 then '0'||try_cast(i.MES as varchar) else try_cast(i.MES as varchar) end MESDATPER,
	i.ANO,
	i.MES,
	i.SERVICO,
	i.PRESTADORA,
	i.UF,
	i.MUNICIPIO NOME_DO_MUNICIPIO,
	i.CODIGO_IBGE,
	i.INDICADOR IND,
	i.RESULTADO,
	i.LIMITE_INFERIOR LIM_INFERIOR,
	i.LIMITE_SUPERIOR LIM_SUPERIOR,
	i.ERRO_AMOSTRAL,
	i.NUMERO_DE_MEDIDAS NUMERO_MEDIDAS,
	i.NUMERO_DE_COLETORES NUMERO_COLETORES
	from
	TABELA_CSV_INDICADORES_RQUAL i 
	where i.SERVICO = 'Telefonia Móvel'
	and i.TIPO = 'Indicador IQS'
)
pivot(
    MAX(RESULTADO),
    MAX(NUMERO_MEDIDAS),
    MAX(NUMERO_COLETORES)
    for ind in
    (
       'IND1' IND1,
       'IND2' IND2,
       'IND3' IND3,
       'IND4' IND4,
       'IND5' IND5,
       'IND6' IND6,
       'IND7' IND7,
       'IND8' IND8
    )
)
 group by MESDATPER, ANO, MES, SERVICO, PRESTADORA, CODIGO_IBGE, UF, NOME_DO_MUNICIPIO
 order by MESDATPER, UF, CODIGO_IBGE
;
'''

conn.execute(v_sql_smp)

v_tbl_smp = '''
select MESDATPER, ANO, MES, REPLACE(SERVICO, 'Telefonia Móvel', 'SMP') SERVICO, PRESTADORA, CODIGO_IBGE, UF, NOME_DO_MUNICIPIO,
IND1, IND2, IND3, IND4, IND5, IND6, IND7, IND8, NUMERO_MEDIDAS, NUMERO_COLETORES, VALIDADE_ESTATISTICA
from V_RQUAL_ANATEL_SMP
'''

df_rqual_smp = conn.execute(v_tbl_smp).df()
df_estacoes_smp = conn.execute('select * from ESTACOES_SMP').df()

df_ibge = conn.execute('select * from DTB_IBGE_MUNICIPIOS').df()
df_arealocal = conn.execute('select * from AREALOCAL').df()

conn.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Criação da camada _Silver_ no Spark e realizando a ingestão dos dados;

Estamos definindo em sequencia duas Tabelas como "Fatos" e duas dimensões (IBGE e Area Local)

In [0]:
%sql 
DROP DATABASE IF EXISTS l_silver CASCADE;
CREATE DATABASE l_silver;

In [0]:
#Cria tabela F_RQUAL_SMP
rqual_smp_spark_df = spark.createDataFrame(df_rqual_smp)
rqual_smp_spark_df.write.mode("overwrite").saveAsTable("l_silver.f_rqual_smp")

#Cria tabela F_ESTACOES_SMP
estacoes_smp_spark_df = spark.createDataFrame(df_estacoes_smp)
estacoes_smp_spark_df.write.mode("overwrite").saveAsTable("l_silver.f_estacoes_smp")

#Cria tabela D_IBGE
df_ibge_subset = df_ibge[['CODIGO_MUNICIPIO_COMPLETO', 'NOME_MUNICIPIO', 'UF', 'NOME_UF']].copy()
df_ibge_subset.columns = ['COD_IBGE', 'MUNICIPIO', 'COD_UF', 'UF']
ibge_spark_df = spark.createDataFrame(df_ibge_subset)
ibge_spark_df.write.mode("overwrite").saveAsTable("l_silver.d_ibge")

#Cria tabela D_AREALOCAL
df_arealocal_subset = df_arealocal[['CO_MUNICIPIO_IBGE', 'CN']].copy()
df_arealocal_subset.columns = ['COD_IBGE', 'CN']
arealocal_spark_df = spark.createDataFrame(df_arealocal_subset)
arealocal_spark_df.write.mode("overwrite").saveAsTable("l_silver.d_arealocal")

## Criação da camada _Gold_  no Spark e realizando a ingestão dos dados;

Estamos definindo em sequencia duas Tabelas como "Fatos" e duas dimensões (IBGE e Area Local)

In [0]:
%sql 
DROP DATABASE IF EXISTS l_gold CASCADE;
CREATE DATABASE l_gold;

### Criação da view Erbs_BR - Resultado consolidado de ERBs a nível nacional

In [0]:
%sql 
/* Criar visualização do resultado em agregação a nível País - Brasil */
create or replace view l_gold.vw_erbs_br as
  select *
  from
  (
    SELECT PRESTADORA,
                  TCN,
                  SUM(QTD_ESTAC) AS ERBS
            FROM
              (SELECT DISTINCT
                  upper(bs.EMPRESA_ESTACAO) PRESTADORA,
                  count(distinct bs.numero_estacao) QTD_ESTAC,
                  bs.geracao TCN,
                  al.CN COD_AREA,
                  ib.COD_UF,
                  ib.UF,
                  ib.MUNICIPIO CIDADE,
                  ib.COD_IBGE
                  FROM l_silver.f_estacoes_smp bs
                    join l_silver.d_ibge ib on bs.CODIGO_IBGE = ib.COD_IBGE
                    left join l_silver.d_arealocal al on bs.CODIGO_IBGE = al.COD_IBGE
                  where bs.geracao is not null
                  group by
                  bs.EMPRESA_ESTACAO, bs.GERACAO, al.CN, ib.COD_UF, ib.UF, ib.MUNICIPIO, ib.COD_IBGE
                )
          group by PRESTADORA, TCN
  )
  pivot(
    max(erbs) AS qtd_erbs
        FOR tcn IN ('2G' AS TCN_2G, '3G' AS TCN_3G, '4G' AS TCN_4G, '5G' AS TCN_5G)
      )
;

In [0]:
%sql
select * from l_gold.vw_erbs_br
limit 10
;

PRESTADORA,TCN_2G,TCN_3G,TCN_4G,TCN_5G
TIM,18271,19981,31815,14969
VIVO,16193,26621,35017,18181
CLARO,19965,24941,29149,13672
BRISANET,null,null,2145,1809
SERCOMTEL,38,42,null,1
GIGA+,null,null,12,null
UNIFIQUE TELECOMUNICACOES S/A,null,null,319,285
IEZ! TELECOM LTDA.,null,null,19,19
ALGAR,395,614,566,191


### Criação da view Erbs_UF - Resultado consolidado de ERBs a nível de unidades federativas (estados da união)

In [0]:
%sql 
/* Criar visualização do resultado em agregação a nível UF - Brasil */
create or replace view l_gold.vw_erbs_uf as
  select *
  from
  (
    SELECT 
          UF, COD_UF, COD_AREA,
          PRESTADORA,
          TCN,
          SUM(QTD_ESTAC) AS ERBS
            FROM
              (SELECT DISTINCT
                  upper(bs.EMPRESA_ESTACAO) PRESTADORA,
                  count(distinct bs.numero_estacao) QTD_ESTAC,
                  bs.geracao TCN,
                  al.CN COD_AREA,
                  ib.COD_UF,
                  ib.UF,
                  ib.MUNICIPIO CIDADE,
                  ib.COD_IBGE
                  FROM l_silver.f_estacoes_smp bs
                    join l_silver.d_ibge ib on bs.CODIGO_IBGE = ib.COD_IBGE
                    left join l_silver.d_arealocal al on bs.CODIGO_IBGE = al.COD_IBGE
                  where bs.geracao is not null
                  group by
                  bs.EMPRESA_ESTACAO, bs.GERACAO, al.CN, ib.COD_UF, ib.UF, ib.MUNICIPIO, ib.COD_IBGE
                )
          group by 
          UF, COD_UF, COD_AREA,
          PRESTADORA, TCN
  )
  pivot(
    max(erbs) AS qtd_erbs
        FOR tcn IN ('2G' AS TCN_2G, '3G' AS TCN_3G, '4G' AS TCN_4G, '5G' AS TCN_5G)
      )
;

In [0]:
%sql
select * from l_gold.vw_erbs_uf
limit 10
;

UF,COD_UF,COD_AREA,PRESTADORA,TCN_2G,TCN_3G,TCN_4G,TCN_5G
Goiás,52,62,VIVO,213,375,650,422
Goiás,52,62,CLARO,489,601,703,370
Goiás,52,62,TIM,269,315,791,407
Amapá,16,96,VIVO,60,127,153,125
Amapá,16,96,TIM,70,69,108,85
Amapá,16,96,CLARO,45,69,101,79
Pernambuco,26,87,TIM,156,155,192,28
Pernambuco,26,87,VIVO,72,245,282,49
Pernambuco,26,87,CLARO,161,181,196,42
Pernambuco,26,87,BRISANET,null,null,121,75


### Criação da view Erbs_Cid - Resultado consolidado de ERBs a nível dos municípios das unidades federativas (Cidades)

In [0]:
%sql 
/* Criar visualização do resultado em agregação a nível Cidades - Brasil */
create or replace view l_gold.vw_erbs_cid as
  select *
  from
  (
    SELECT 
          UF, COD_UF, COD_AREA,
          COD_IBGE, CIDADE, CAPITAL,
          PRESTADORA,
          TCN,
          SUM(QTD_ESTAC) AS ERBS
            FROM
              (SELECT DISTINCT
                  upper(bs.EMPRESA_ESTACAO) PRESTADORA,
                  count(distinct bs.numero_estacao) QTD_ESTAC,
                  bs.geracao TCN,
                  al.CN COD_AREA,
                  ib.COD_UF,
                  ib.UF,
                  ib.MUNICIPIO CIDADE,
                  ib.COD_IBGE,
                  case
                  when ib.COD_IBGE in (
                    1100205,1302603,1200401,5002704,1600303,
                    5300108,1400100,5103403,1721000,3550308,
                    2211001,3304557,1501402,5208707,2927408,
                    4205407,2111300,2704302,4314902,4106902,
                    3106200,2304400,2611606,2507507,2800308,
                    2408102,3205309
                  ) then
                  'SIM' else 'NÃO' end as CAPITAL
                  FROM l_silver.f_estacoes_smp bs
                    join l_silver.d_ibge ib on bs.CODIGO_IBGE = ib.COD_IBGE
                    left join l_silver.d_arealocal al on bs.CODIGO_IBGE = al.COD_IBGE
                  where bs.geracao is not null
                  group by
                  bs.EMPRESA_ESTACAO, bs.GERACAO, al.CN, ib.COD_UF, ib.UF, ib.MUNICIPIO, ib.COD_IBGE, CAPITAL
                )
          group by 
          UF, COD_UF, COD_AREA,
          COD_IBGE, CIDADE, CAPITAL,
          PRESTADORA, TCN
  )
  pivot(
    max(erbs) AS qtd_erbs
        FOR tcn IN ('2G' AS TCN_2G, '3G' AS TCN_3G, '4G' AS TCN_4G, '5G' AS TCN_5G)
      )
;

In [0]:
%sql
select * from l_gold.vw_erbs_cid
limit 10
;

UF,COD_UF,COD_AREA,COD_IBGE,CIDADE,CAPITAL,PRESTADORA,TCN_2G,TCN_3G,TCN_4G,TCN_5G
Goiás,52,62,5208707,Goiânia,SIM,VIVO,74,173,237,226
Goiás,52,62,5208707,Goiânia,SIM,CLARO,177,236,266,226
Goiás,52,62,5208707,Goiânia,SIM,TIM,110,141,335,240
Goiás,52,62,5201108,Anápolis,NÃO,CLARO,54,68,71,40
Goiás,52,62,5201108,Anápolis,NÃO,TIM,19,36,47,31
Goiás,52,62,5201108,Anápolis,NÃO,VIVO,12,22,49,45
Goiás,52,62,5205802,Corumbá de Goiás,NÃO,CLARO,2,4,4,null
Goiás,52,62,5205802,Corumbá de Goiás,NÃO,TIM,2,3,7,null
Goiás,52,62,5214903,Nova Roma,NÃO,CLARO,1,1,1,null
Goiás,52,62,5214903,Nova Roma,NÃO,VIVO,1,1,1,null


### Criação da view Indicadores - Resultado do Plano de Melhoria de Qualidade divulgado pela Anatel
Estaremos utilizando um recorte do período a partir do primeiro semestre de 2025 em diante.

In [0]:
%sql
create or replace table l_gold.vw_indicadores as
select 
MESDATPER, 
ind.ANO, ind.MES, ind.SERVICO, ind.PRESTADORA,
al.CN, ib.COD_IBGE, ib.UF, ib.COD_UF, ib.MUNICIPIO CIDADE, 
ind.IND1, ind.IND2, ind.IND3, ind.IND4, ind.IND5, ind.IND6, ind.IND7, ind.IND8,
ind.NUMERO_MEDIDAS, ind.NUMERO_COLETORES, ind.VALIDADE_ESTATISTICA
    FROM
        l_silver.f_rqual_smp ind
        join l_silver.d_ibge ib on ind.CODIGO_IBGE = ib.COD_IBGE
        left join l_silver.d_arealocal al on ind.CODIGO_IBGE = al.COD_IBGE
where mesdatper >= 202501
;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from l_gold.vw_indicadores
limit 10
;

MESDATPER,ANO,MES,SERVICO,PRESTADORA,CN,COD_IBGE,UF,COD_UF,CIDADE,IND1,IND2,IND3,IND4,IND5,IND6,IND7,IND8,NUMERO_MEDIDAS,NUMERO_COLETORES,VALIDADE_ESTATISTICA
202501,2025,1,SMP,TIM,68,1200013,Acre,12,Acrelândia,"99,9188","0,5949","99,8196","88,8888","80,0000","86,6666","86,6666","100,0000",12,8,0
202501,2025,1,SMP,CLARO,68,1200013,Acre,12,Acrelândia,"99,8851","0,0625","99,7570","91,4096","91,3627","87,9078","96,5517","100,0000",453,100,1
202501,2025,1,SMP,VIVO,68,1200013,Acre,12,Acrelândia,"99,8655","0,2682","99,8852","100,0000","80,0000","100,0000","100,0000","100,0000",4,1,0
202501,2025,1,SMP,CLARO,68,1200054,Acre,12,Assis Brasil,"99,7636","0,1621","99,5206","82,0083","81,3953","84,4961","97,6923","100,0000",119,50,1
202501,2025,1,SMP,VIVO,68,1200054,Acre,12,Assis Brasil,"99,7095","0,6333","99,7377",NI,NI,NI,NI,"100,0000",null,null,0
202501,2025,1,SMP,TIM,68,1200054,Acre,12,Assis Brasil,"99,7854","0,9037","99,2096",NA,NA,NA,NA,"99,9708",null,null,0
202501,2025,1,SMP,VIVO,68,1200104,Acre,12,Brasiléia,"99,8879","0,1179","99,9177","97,7777","76,1904","100,0000","69,3877","99,3010",40,21,0
202501,2025,1,SMP,TIM,68,1200104,Acre,12,Brasiléia,"99,7822","0,5099","99,9212","95,6521","91,6666","91,6666","91,6666","100,0000",11,6,0
202501,2025,1,SMP,CLARO,68,1200104,Acre,12,Brasiléia,"99,8019","0,0708","99,6897","94,5475","89,0041","84,2323","96,0416","100,0000",431,217,1
202501,2025,1,SMP,TIM,68,1200138,Acre,12,Bujari,"99,8235","0,7618","99,6320","90,9090","88,2352","94,1176","100,0000","100,0000",16,9,0
